# Supplemental Figure 3a - Uphill mutational path overlap analysis for each 'strong' model (affinity-only, antigen-capture, and competitive-capture)

- This script computes how many complete germline-to-somatic paths(discrete mutation orderings) are fully uphill under each model and their
intersections by using an element-wise minimum trick on the unnormalized transition matrices, so no explicit path enumeration is needed

In [2]:
import numpy as np
from numpy.random import SeedSequence
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib_venn import venn3, venn3_circles
from scipy.sparse import dok_matrix, csr_matrix
from tqdm import tqdm
import os

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Arial'
plt.rcParams['mathtext.it'] = 'Arial:italic'
plt.rcParams['mathtext.bf'] = 'Arial:bold'
mpl.rcParams['figure.dpi'] = 2000
plt.rcParams.update({
    'font.size': 7,
    'axes.linewidth': 0.5,
    'xtick.major.size': 2, 'xtick.major.width': 0.5,
    'ytick.major.size': 2, 'ytick.major.width': 0.5,
})

# configuration 
ANTIBODY = 'omi-32'
SCENARIO = 'ba1'
L = 13
ANTIGENS = ['wuhan', 'ba1', 'ba4']

SELECTION_STRENGTH = 'strong'
GERMLINE_NONBINDING = False
NORMALIZE_PHENOTYPES = True

# concentrations for capture models (log-linear regime, see Methods)
ANTIGEN_CONCENTRATION = 1e-20
PSR_CONCENTRATION = 1e10

# bootstrap config
NBOOTSTRAP = 10
BOOTSTRAP_MASTER_SEED = 42
_ss = SeedSequence(BOOTSTRAP_MASTER_SEED)
_child_seeds = _ss.spawn(NBOOTSTRAP)

# model definitions
MODEL_NAMES = ['affinity', 'capture', 'competitive_capture']
MODEL_LABELS = ['affinity', 'antigen capture', 'competitive capture']
COLORS = ['#DC143C', '#FFD600', '#0000CD']

ALPHA_SINGLE = 0.25
ALPHA_PAIR = 0.35
ALPHA_TRIPLE = 0.4

DATA_ROOT = '../../epistasis_inference/linear_interaction_models'

PHENOTYPE_FILES = {
    'wuhan': f'{DATA_ROOT}/wuhan/reference-based/predicted_phenotypes/wuhan_raw_2order_full_biochem_predictions.csv',
    'ba1':   f'{DATA_ROOT}/ba1/reference-based/predicted_phenotypes/ba1_raw_3order_full_biochem_predictions.csv',
    'ba4':   f'{DATA_ROOT}/ba4/reference-based/predicted_phenotypes/ba4_raw_3order_full_biochem_predictions.csv',
    'expression': f'{DATA_ROOT}/expression/reference-based/predicted_phenotypes/expression_raw_2order_full_biochem_predictions.csv',
    'psr':   f'{DATA_ROOT}/psr/reference-based/predicted_phenotypes/psr_raw_2order_full_biochem_predictions.csv',
}


# helper functions
sequences_int = range(2**L)

def nb_mutation(s):
    return bin(s).count("1")

def reachable(s, L):
    return [(s | 2**ii) for ii in range(L) if (s | 2**ii) > s]


# load and prepare data
print("Loading phenotype predictions...")

def load_phenotype(path, prefix):
    d = pd.read_csv(path)
    d['geno'] = d['geno'].astype('string').str.zfill(L)
    d[f'{prefix}_mean'] = d['predicted_phenotype']
    d[f'{prefix}_sem'] = d['prediction_se']
    return d[['geno', f'{prefix}_mean', f'{prefix}_sem']]

wuhan = load_phenotype(PHENOTYPE_FILES['wuhan'], 'wuhan_log10kd')
ba1   = load_phenotype(PHENOTYPE_FILES['ba1'], 'ba1_log10kd')
ba4   = load_phenotype(PHENOTYPE_FILES['ba4'], 'ba4_log10kd')
expr  = load_phenotype(PHENOTYPE_FILES['expression'], 'mednorm')
psr   = load_phenotype(PHENOTYPE_FILES['psr'], 'ec50')

df = wuhan.merge(ba1, on='geno', how='outer')
df = df.merge(ba4, on='geno', how='outer')
df = df.merge(expr, on='geno', how='outer')
df = df.merge(psr, on='geno', how='outer')
df['variant'] = df['geno']

df = df.rename(columns={
    "wuhan_log10kd_mean": "wuhan_log10Kd",
    "ba1_log10kd_mean": "ba1_log10Kd",
    "ba4_log10kd_mean": "ba4_log10Kd",
    "wuhan_log10kd_sem": "wuhan_stelog10Kd",
    "ba1_log10kd_sem": "ba1_stelog10Kd",
    "ba4_log10kd_sem": "ba4_stelog10Kd",
    "mednorm_mean": "expression_log10",
    "mednorm_sem": "expression_stelog10",
    "ec50_mean": "psr_neglog10ec50",
    "ec50_sem": "psr_stelog10ec50",
})

df["variant_int"] = df.variant.apply(lambda x: int(x, 2))
df = df.set_index("variant_int").reindex(range(0, 2**L)).reset_index()
df['variant'] = df['variant'].fillna(
    df['variant_int'].apply(lambda x: format(x, f'0{L}b'))
)
df['mixed_log10Kd'] = df[[f"{a}_log10Kd" for a in ANTIGENS]].mean(axis=1)

print(f"  Loaded {len(df)} variants")


# normalization
print("Computing normalization factors...")

affinity_ranges = {
    ag: df[f"{ag}_log10Kd"].max() - df[f"{ag}_log10Kd"].min()
    for ag in ANTIGENS
}
TARGET_RANGE = np.mean(list(affinity_ranges.values()))

EXPR_MIN = df['expression_log10'].min()
EXPR_RANGE = df['expression_log10'].max() - EXPR_MIN
PSR_MIN = df['psr_neglog10ec50'].min()
PSR_RANGE = df['psr_neglog10ec50'].max() - PSR_MIN

if NORMALIZE_PHENOTYPES:
    EXPR_SCALE = TARGET_RANGE / EXPR_RANGE
    PSR_SCALE = TARGET_RANGE / PSR_RANGE
    print(f"  Expression scale: {EXPR_SCALE:.3f}x, PSR scale: {PSR_SCALE:.3f}x")
else:
    EXPR_SCALE = 1.0
    PSR_SCALE = 1.0


# fitness calculation
def calculate_fitness(ag, fitness_model, bootstrap_replicate=None):
    if bootstrap_replicate is not None:
        if ag != 'mixed':
            affinity = np.random.normal(
                df[f"{ag}_log10Kd"].values.astype('float32'),
                df[f"{ag}_stelog10Kd"].values.astype('float32'),
            )
        else:
            affinity = sum([
                np.random.normal(
                    df[f"{aa}_log10Kd"].values.astype('float32'),
                    df[f"{aa}_stelog10Kd"].values.astype('float32'),
                ) for aa in ANTIGENS
            ]) / len(ANTIGENS)

        expression = np.random.normal(
            df['expression_log10'].values.astype('float32'),
            df['expression_stelog10'].values.astype('float32'),
        )
        psr = np.random.normal(
            df['psr_neglog10ec50'].values.astype('float32'),
            df['psr_stelog10ec50'].values.astype('float32'),
        )
    else:
        if ag != 'mixed':
            affinity = df[f"{ag}_log10Kd"].values.astype('float32')
        else:
            affinity = df['mixed_log10Kd'].values.astype('float32')
        expression = df['expression_log10'].values.astype('float32')
        psr = df['psr_neglog10ec50'].values.astype('float32')

    if NORMALIZE_PHENOTYPES:
        expression_scaled = EXPR_MIN + (expression - EXPR_MIN) * EXPR_SCALE
        psr_scaled = PSR_MIN + (psr - PSR_MIN) * PSR_SCALE
    else:
        expression_scaled = expression
        psr_scaled = psr

    if fitness_model == 'affinity':
        return affinity

    elif fitness_model == 'capture':
        expression_linear = 10**expression_scaled
        kd_linear = 10**(-affinity)
        fraction_bound = ANTIGEN_CONCENTRATION / (ANTIGEN_CONCENTRATION + kd_linear)
        return np.log10(expression_linear * fraction_bound)

    elif fitness_model == 'competitive_capture':
        expression_linear = 10**expression_scaled
        kd_linear = 10**(-affinity)
        ec50_linear = 10**(-psr_scaled)
        penalty_term = 1 + PSR_CONCENTRATION / ec50_linear
        denominator = ANTIGEN_CONCENTRATION + kd_linear * penalty_term
        fraction_bound = ANTIGEN_CONCENTRATION / denominator
        return np.log10(expression_linear * fraction_bound)

    else:
        raise ValueError(f"Unknown fitness_model: {fitness_model}")


def fixation_probability_binary(s, fitness_s, fitness_t):
    if nb_mutation(s) == 0 and GERMLINE_NONBINDING:
        return 1.0
    elif np.isnan(fitness_s) or np.isnan(fitness_t):
        return 0.0
    elif (fitness_t - fitness_s) > 0:
        return 1.0
    else:
        return 0.0


# build transition matrices
def build_unnormed_matrix(ag, fitness_model, bootstrap_replicate=None):
    fitnesses = calculate_fitness(ag, fitness_model,
                                  bootstrap_replicate=bootstrap_replicate)
    mat = dok_matrix((2**L, 2**L), dtype=np.float64)
    for s in sequences_int:
        for t in reachable(s, L):
            if fixation_probability_binary(s, fitnesses[s], fitnesses[t]) > 0:
                mat[s, t] = 1.0
    return mat.tocsr()


def count_uphill_paths(matrix, L):
    P = csr_matrix(np.zeros((1, 2**L)))
    P[0, 0] = 1.0
    for _ in range(L):
        P = P @ matrix
    return int(np.array(P.todense()).flatten()[2**L - 1])


def elementwise_min(*matrices):
    result = matrices[0].copy()
    for m in matrices[1:]:
        result = result.minimum(m)
    return result


# calculate the number of uphill paths for each selection model
print(f"\nBuilding transition matrices for {SCENARIO} scenario...")
print(f"  {NBOOTSTRAP} bootstrap replicates x 3 models\n")

ag = SCENARIO

region_keys = ['A_only', 'B_only', 'C_only',
               'AB_only', 'AC_only', 'BC_only', 'ABC']
region_counts = {r: [] for r in region_keys}

for nb in tqdm(range(NBOOTSTRAP), desc="Bootstrap replicates"):
    seed_val = _child_seeds[nb].generate_state(1)[0]

    M = {}
    for model_name in MODEL_NAMES:
        np.random.seed(seed_val)
        # build all antigens in original order to match random state
        for ag_iter in ANTIGENS + ['mixed']:
            mat = build_unnormed_matrix(ag_iter, model_name,
                                        bootstrap_replicate=nb)
            if ag_iter == SCENARIO:
                M[model_name] = mat

    A = M['affinity']
    B = M['capture']
    C = M['competitive_capture']

    # intersection matrices via element-wise minimum
    AB = elementwise_min(A, B)
    AC = elementwise_min(A, C)
    BC = elementwise_min(B, C)
    ABC_mat = elementwise_min(A, B, C)

    # count paths for each
    n_A   = count_uphill_paths(A, L)
    n_B   = count_uphill_paths(B, L)
    n_C   = count_uphill_paths(C, L)
    n_AB  = count_uphill_paths(AB, L)
    n_AC  = count_uphill_paths(AC, L)
    n_BC  = count_uphill_paths(BC, L)
    n_ABC = count_uphill_paths(ABC_mat, L)

    # inclusion-exclusion for Venn regions
    region_counts['ABC'].append(n_ABC)
    region_counts['AB_only'].append(n_AB - n_ABC)
    region_counts['AC_only'].append(n_AC - n_ABC)
    region_counts['BC_only'].append(n_BC - n_ABC)
    region_counts['A_only'].append(n_A - n_AB - n_AC + n_ABC)
    region_counts['B_only'].append(n_B - n_AB - n_BC + n_ABC)
    region_counts['C_only'].append(n_C - n_AC - n_BC + n_ABC)

    if nb == 0:
        print(f"\n  Bootstrap 0 check: A={n_A:,}, B={n_B:,}, C={n_C:,}")
        print(f"    AB={n_AB:,}, AC={n_AC:,}, BC={n_BC:,}, ABC={n_ABC:,}")


# summary 
means = {r: np.mean(v) for r, v in region_counts.items()}
sems  = {r: np.std(v, ddof=1) / np.sqrt(NBOOTSTRAP)
         for r, v in region_counts.items()}
total_paths = sum(means.values())

print("\n" + "=" * 60)
print("PATH-LEVEL OVERLAP (mean +/- SEM across bootstraps)")
print("=" * 60)
for r in region_keys:
    pct = 100 * means[r] / total_paths if total_paths > 0 else 0
    print(f"  {r:12s}: {means[r]:>14,.1f} +/- {sems[r]:>10,.1f}  ({pct:5.1f}%)")
print(f"  {'TOTAL':12s}: {total_paths:>14,.1f}")
print("=" * 60)


# Venn diagram using precise circle area overlap (circle area proportional to number of paths for each individual selection model)
print("\nPlotting Venn diagram...")

CANVAS_W = 1.2
CANVAS_H = 1.2
LEFT_MARGIN = 0.15
RIGHT_MARGIN = 0.15
TOP_MARGIN = 0.15
BOTTOM_MARGIN = 0.15

fig_w = LEFT_MARGIN + CANVAS_W + RIGHT_MARGIN
fig_h = TOP_MARGIN + CANVAS_H + BOTTOM_MARGIN

fig = plt.figure(figsize=(fig_w, fig_h))
ax = fig.add_axes([
    LEFT_MARGIN / fig_w,
    BOTTOM_MARGIN / fig_h,
    CANVAS_W / fig_w,
    CANVAS_H / fig_h,
])

# venn3 subset order: (A, B, AB, C, AC, BC, ABC)
subsets = (
    means['A_only'],
    means['B_only'],
    means['AB_only'],
    means['C_only'],
    means['AC_only'],
    means['BC_only'],
    means['ABC'],
)

v = venn3(subsets=subsets, set_labels=('', '', ''), ax=ax)

patch_colors = {
    '100': ('#DC143C', ALPHA_SINGLE),
    '010': ('#FFD600', ALPHA_SINGLE),
    '001': ('#0000CD', ALPHA_SINGLE),
    '110': ('#FF6B00', ALPHA_PAIR),
    '101': ('#6A0080', ALPHA_PAIR),
    '011': ('#007D00', ALPHA_PAIR),
    '111': ('#888888', ALPHA_TRIPLE),
}

for pid, (col, alpha) in patch_colors.items():
    patch = v.get_patch_by_id(pid)
    if patch:
        patch.set_color(col)
        patch.set_alpha(alpha)
        patch.set_edgecolor(col)
        patch.set_linewidth(0.75)

region_map = {
    '100': 'A_only', '010': 'B_only', '001': 'C_only',
    '110': 'AB_only', '101': 'AC_only', '011': 'BC_only',
    '111': 'ABC',
}
for pid, region in region_map.items():
    lbl = v.get_label_by_id(pid)
    if lbl:
        val = means[region]
        pct = 100 * val / total_paths if total_paths > 0 else 0
        lbl.set_text(f"{val:,.0f}\n({pct:.0f}%)")
        lbl.set_fontsize(7)
        lbl.set_color('#222222')

c = venn3_circles(subsets=subsets, ax=ax, linewidth=0.75)
for circle, col in zip(c, COLORS):
    circle.set_edgecolor(col)
    circle.set_linewidth(0.75)

centres = [v.get_circle_center(i) for i in range(3)]
radii = [v.get_circle_radius(i) for i in range(3)]

label_configs = [
    (centres[0].x, centres[0].y + radii[0] + 0.06,
     MODEL_LABELS[0], COLORS[0], 'center', 'bottom'),
    (centres[1].x, centres[1].y + radii[1] + 0.06,
     MODEL_LABELS[1], COLORS[1], 'center', 'bottom'),
    (centres[2].x, centres[2].y - radii[2] - 0.06,
     MODEL_LABELS[2], COLORS[2], 'center', 'top'),
]

for (x, y, txt, col, ha, va) in label_configs:
    ax.text(x, y, txt, ha=ha, va=va, fontsize=8,
            color=col, fontweight='bold')

out_stem = f"S_Figure_3a"
plt.savefig(f"{out_stem}.pdf", bbox_inches='tight', transparent=True)
print(f"\n  Saved {out_stem}.pdf")
plt.close()

print("\nDone.")


Loading phenotype predictions...
  Loaded 8192 variants
Computing normalization factors...
  Expression scale: 5.678x, PSR scale: 2.643x

Building transition matrices for ba1 scenario...
  10 bootstrap replicates x 3 models



Bootstrap replicates:   0%|          | 0/10 [00:00<?, ?it/s]/opt/anaconda3/envs/epistasis/lib/python3.13/site-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])
Bootstrap replicates:  10%|█         | 1/10 [00:01<00:13,  1.45s/it]


  Bootstrap 0 check: A=901,138, B=298,523, C=8,516
    AB=41, AC=0, BC=40, ABC=0


Bootstrap replicates: 100%|██████████| 10/10 [00:14<00:00,  1.45s/it]



PATH-LEVEL OVERLAP (mean +/- SEM across bootstraps)
  A_only      :      480,287.4 +/-   90,687.0  ( 60.9%)
  B_only      :      135,428.0 +/-   33,445.2  ( 17.2%)
  C_only      :      173,239.0 +/-   45,417.4  ( 22.0%)
  AB_only     :           18.7 +/-        6.6  (  0.0%)
  AC_only     :           75.5 +/-       35.3  (  0.0%)
  BC_only     :          160.9 +/-       84.5  (  0.0%)
  ABC         :            0.0 +/-        0.0  (  0.0%)
  TOTAL       :      789,209.5

Plotting Venn diagram...

  Saved S_Figure_3a.png

  Saved S_Figure_3a.pdf

Done.
